# Module 05: Random Boolean Networks & Distributed Immune Selection
Exploration of **Kauffman Random Boolean Networks (RBNs), Derrida Annealing Curves, Synchronous vs. Asynchronous Attractors (Samuelsson & Troein, 2003; Greil & Drossel, 2005), Gene Knockouts, and Independent Immune Clonal Selection & Kinetic Proofreading**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. KAUFFMAN RANDOM BOOLEAN NETWORK (RBN) ENGINE
# ------------------------------------------------------------------------------

def generate_rbn(n_nodes=10, k_inputs=2, bias_p=0.5, seed=42):
    '''Generates an RBN with N nodes, in-degree K, and Boolean output bias p.'''
    rng = np.random.default_rng(seed)
    wiring = np.zeros((n_nodes, k_inputs), dtype=np.int32)
    for i in range(n_nodes):
        choices = np.delete(np.arange(n_nodes), i)
        wiring[i] = rng.choice(choices, size=min(k_inputs, n_nodes - 1), replace=False)
    truth_tables = (rng.random((n_nodes, 2 ** k_inputs)) < bias_p).astype(np.uint8)
    return wiring, truth_tables

def step_rbn_sync(states, wiring, truth_tables, knockout_node=-1):
    '''Vectorized synchronous update over batch of states (shape: [batch, N]).'''
    batch_size, n_nodes = states.shape
    k_inputs = wiring.shape[1]
    next_states = np.zeros_like(states)
    weights = (1 << np.arange(k_inputs - 1, -1, -1)).astype(np.int32)
    for i in range(n_nodes):
        if i == knockout_node:
            next_states[:, i] = 0
            continue
        inp = states[:, wiring[i]]
        idx = inp.dot(weights)
        next_states[:, i] = truth_tables[i, idx]
    return next_states

def compute_derrida_curve(wiring, truth_tables, n_samples=120, seed=42):
    '''Computes empirical Derrida damage spreading curve d(t+1) vs d(t).'''
    rng = np.random.default_rng(seed)
    n_nodes = wiring.shape[0]
    d_in_list, d_out_list = [], []
    for num_flips in range(1, n_nodes + 1):
        s1 = rng.integers(0, 2, size=(n_samples, n_nodes), dtype=np.uint8)
        s2 = s1.copy()
        for b in range(n_samples):
            flip_idx = rng.choice(n_nodes, size=num_flips, replace=False)
            s2[b, flip_idx] ^= 1
        s1_next = step_rbn_sync(s1, wiring, truth_tables)
        s2_next = step_rbn_sync(s2, wiring, truth_tables)
        d_in = num_flips / float(n_nodes)
        d_out = np.mean(np.abs(s1_next.astype(np.int32) - s2_next.astype(np.int32)))
        d_in_list.append(d_in)
        d_out_list.append(d_out)
    return np.array(d_in_list), np.array(d_out_list)

def analyze_exact_state_space(wiring, truth_tables, knockout_node=-1):
    '''Exhaustively enumerates all 2^N states for synchronous attractors and basins.'''
    n_nodes = wiring.shape[0]
    num_states = 1 << n_nodes
    all_states = ((np.arange(num_states)[:, None] & (1 << np.arange(n_nodes - 1, -1, -1))) > 0).astype(np.uint8)
    next_states = step_rbn_sync(all_states, wiring, truth_tables, knockout_node=knockout_node)
    weights = (1 << np.arange(n_nodes - 1, -1, -1)).astype(np.int32)
    next_idx = next_states.dot(weights)

    visited = np.full(num_states, -1, dtype=np.int32)
    attractors = []
    basin_sizes = []
    for start in range(num_states):
        if visited[start] != -1:
            continue
        path = []
        curr = start
        while visited[curr] == -1:
            visited[curr] = -2
            path.append(curr)
            curr = int(next_idx[curr])
        if visited[curr] == -2:
            cycle_start = path.index(curr)
            cycle = path[cycle_start:]
            att_id = len(attractors)
            attractors.append(cycle)
            basin_sizes.append(len(path))
            for node in path:
                visited[node] = att_id
        else:
            att_id = visited[curr]
            basin_sizes[att_id] += len(path)
            for node in path:
                visited[node] = att_id
    return attractors, basin_sizes

def sample_asynchronous_trajectories(wiring, truth_tables, n_trials=120, steps=160, seed=42):
    '''Simulates random asynchronous updates to contrast with synchronous limit cycles.'''
    rng = np.random.default_rng(seed)
    n_nodes, k_inputs = wiring.shape
    weights = (1 << np.arange(k_inputs - 1, -1, -1)).astype(np.int32)
    states = rng.integers(0, 2, size=(n_trials, n_nodes), dtype=np.uint8)
    for _ in range(steps):
        update_nodes = rng.integers(0, n_nodes, size=n_trials)
        for tr in range(n_trials):
            u = update_nodes[tr]
            inp = states[tr, wiring[u]]
            idx = int(inp.dot(weights))
            states[tr, u] = truth_tables[u, idx]
    # Check how many trials settled into strict point attractors vs loose fluctuating sets
    sync_next = step_rbn_sync(states, wiring, truth_tables)
    is_fixed = np.all(states == sync_next, axis=1)
    return float(np.mean(is_fixed))

# ------------------------------------------------------------------------------
# 2. INDEPENDENT IMMUNE CLONAL SELECTION & KINETIC PROOFREADING MODEL
# ------------------------------------------------------------------------------

def simulate_immune_selection(n_clones=60, n_steps=50, self_threshold=0.72, seed=42):
    '''
    Simulates distributed immune receptor selection:
    Clones reactive to self-antigens (> self_threshold) are deleted (central tolerance);
    remaining clones undergo affinity maturation against a novel foreign pathogen.
    '''
    rng = np.random.default_rng(seed)
    dim = 16
    self_antigen = rng.random(dim)
    pathogen = 1.0 - self_antigen  # Distinct foreign epitope

    receptors = rng.random((n_clones, dim))
    # Negative selection: purge self-reactive clones
    self_aff = 1.0 - np.mean(np.abs(receptors - self_antigen), axis=1)
    surviving = receptors[self_aff < self_threshold]
    if len(surviving) < 5:
        surviving = receptors[:5]

    mean_path_aff, max_path_aff = [], []
    pop = surviving.copy()
    for _ in range(n_steps):
        aff = 1.0 - np.mean(np.abs(pop - pathogen), axis=1)
        mean_path_aff.append(np.mean(aff))
        max_path_aff.append(np.max(aff))
        # Clonal expansion proportional to affinity + somatic hypermutation
        probs = np.exp(8.0 * aff)
        probs /= np.sum(probs)
        idx = rng.choice(len(pop), size=n_clones, p=probs)
        pop = np.clip(pop[idx] + rng.normal(0, 0.04, size=(n_clones, dim)), 0.0, 1.0)
    return np.array(mean_path_aff), np.array(max_path_aff)

# ------------------------------------------------------------------------------
# 3. INTERACTIVE MODULE 05 DASHBOARD
# ------------------------------------------------------------------------------

def run_module05_lab(mode='1. RBN Dynamics & Derrida Curve', k_inputs=2, bias_p=0.5, knockout_gene=False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    fig.patch.set_facecolor('#F8FAFC')

    if mode == '1. RBN Dynamics & Derrida Curve':
        wiring, tt = generate_rbn(n_nodes=10, k_inputs=k_inputs, bias_p=bias_p, seed=42)
        d_in, d_out = compute_derrida_curve(wiring, tt, n_samples=150, seed=42)
        d_grid = np.linspace(0, 1, 100)
        d_theory = 2.0 * bias_p * (1.0 - bias_p) * (1.0 - (1.0 - d_grid) ** k_inputs)
        sens_s = 2.0 * k_inputs * bias_p * (1.0 - bias_p)

        # Panel 1: Derrida Annealing Curve
        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(d_grid, d_grid, 'k--', alpha=0.5, label='Neutral Line (d_out = d_in)')
        axes[0].plot(d_grid, d_theory, color='#0D9488', linewidth=2.2, label='Annealed Theory')
        axes[0].scatter(d_in, d_out, color='#E11D48', s=55, zorder=5, label='Empirical RBN (N=10)')
        axes[0].set_title(f'1. Derrida Damage Spreading (K={k_inputs}, p={bias_p:.2f} | s={sens_s:.2f})', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Initial Normalized Hamming Distance d(t)')
        axes[0].set_ylabel('Next-Step Distance d(t+1)')
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # Panel 2: Synchronous vs Asynchronous Attractors
        ko_node = 0 if knockout_gene else -1
        atts, basins = analyze_exact_state_space(wiring, tt, knockout_node=ko_node)
        async_fixed_frac = sample_asynchronous_trajectories(wiring, tt, n_trials=120, steps=150, seed=42)
        lengths = [len(a) for a in atts]
        axes[1].set_facecolor('#F8FAFC')
        bars = axes[1].bar(range(1, len(basins) + 1), basins, color='#2563EB', edgecolor='#0F172A')
        for idx_b, (bar, l_val) in enumerate(zip(bars, lengths)):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 12, f'L={l_val}', ha='center', fontsize=8.5, fontweight='bold')
        axes[1].set_ylim(0, 1150)
        axes[1].set_title(f'2. Exact Synchronous Basins (Total={len(atts)} | Async Fixed={async_fixed_frac:.0%})', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Attractor ID (L = Cycle Period)')
        axes[1].set_ylabel('Basin Size (out of 1024 states)')
        axes[1].grid(axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        regime = 'ORDERED (s < 1)' if sens_s < 0.95 else ('CHAOTIC (s > 1)' if sens_s > 1.05 else 'CRITICAL EDGE (s ~ 1)')
        print(f'Regime: {regime} | Sensitivity s = {sens_s:.3f} | Sync Attractors = {len(atts)} | Async Point-Attractor Convergence = {async_fixed_frac:.1%}')

    else:
        # Mode 2: Independent Immune Clonal Selection & McKeithan Kinetic Proofreading
        mean_aff, max_aff = simulate_immune_selection(n_clones=70, n_steps=45, self_threshold=0.72, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(mean_aff, color='#2563EB', linewidth=2.2, label='Population Mean Affinity')
        axes[0].plot(max_aff, color='#D97706', linewidth=2.2, linestyle='--', label='Best Clone Affinity')
        axes[0].set_title('1. Distributed Clonal Selection & Affinity Maturation', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Selection Generation')
        axes[0].set_ylabel('Antigen Binding Affinity')
        axes[0].set_ylim(0.4, 1.02)
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # McKeithan (1995) Kinetic Proofreading Specificity Curve
        steps_n = np.arange(1, 11)
        kp = 1.0
        koff_foreign = 0.25
        koff_self = 1.25  # 5x faster dissociation for self-ligand
        sig_foreign = (kp / (kp + koff_foreign)) ** steps_n
        sig_self = (kp / (kp + koff_self)) ** steps_n
        specificity = sig_foreign / sig_self

        axes[1].set_facecolor('#F8FAFC')
        axes[1].semilogy(steps_n, specificity, marker='o', color='#E11D48', linewidth=2.4)
        axes[1].set_title('2. McKeithan Kinetic Proofreading Specificity', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Number of Proofreading Steps (N)')
        axes[1].set_ylabel('Foreign / Self Discrimination Ratio (Log Scale)')
        axes[1].grid(True, which='both', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print(f'Final Clonal Affinity = {mean_aff[-1]:.3f} | 8-Step Proofreading Specificity Gain = {specificity[7]:.1f}x')

widgets.interact(
    run_module05_lab,
    mode=widgets.Dropdown(options=['1. RBN Dynamics & Derrida Curve', '2. Immune Clonal Selection & Proofreading'], value='1. RBN Dynamics & Derrida Curve', description='Lab Section:'),
    k_inputs=widgets.IntSlider(min=1, max=5, step=1, value=2, description='In-Degree K:'),
    bias_p=widgets.FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5, description='Bias p:'),
    knockout_gene=widgets.Checkbox(value=False, description='Knockout Node 0 (GRN)'),
);
